# s01 - Federated data partition (SURVEY paper, Fase 0)

**Konteks:** notebook ini milik **paper SURVEY** (`paper-survey.md`), BUKAN Paper 4.
Perbedaan utama vs `notebooks/01_*`: kita **TIDAK** menyebut/memakai SFM (metode
kami yang masih review). Penyelarasan skema fitur CIC vs UNSW di sini adalah
**manual feature-name alignment** -- pemetaan nama kolom berpasangan yang **sudah
lazim di literatur cross-dataset** (mis. Lavaur dkk. [F11]). Jadi ini bagian dari
*setup eksperimen*, bukan kontribusi metodologis.

**Tujuan:** tiap *klien* menerapkan tabel padanan fitur ke datanya SENDIRI -> 9 fitur
selaras lokal. Client-CIC & Client-UNSW TIDAK bertukar data; yang disepakati hanya
*skema* (daftar 9 pasangan nama kolom).

**Output:** `survey_out/dataset_stats.csv` (statistik per-klien, ANGKA NYATA) +
partisi per-klien (`survey_out/*.npz`) + unggah ke S3
`unsw-far/federated_survey/data/{cic,unsw}/`.

**PENTING (skala CIC):** `cleaned_100.pkl` menyimpan fitur yang SUDAH di-StandardScaler.
Notebook mengembalikan skala asli (`X_orig = X_scaled * scale_ + mean_`) SEBELUM
alignment agar sepadan dgn UNSW (raw).

> Jalankan di SageMaker/EC2 ber-data. JANGAN mengarang angka.

## 0. Setup & tabel padanan fitur (manual feature-name alignment, metode yang sudah ada)

In [ ]:
import importlib, sys, subprocess
for m in ('pandas','numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, pickle, numpy as np, pandas as pd
OUTDIR='survey_out'; os.makedirs(OUTDIR, exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
S3_PREFIX='unsw-far/federated_survey'  # terpisah dari data Paper 4
SEED=42

# --- Tabel padanan fitur (manual alignment nama kolom). (CIC_col, UNSW_col) ---
# Padanan dasar antar-tool ekstraktor (CICFlowMeter vs Argus/Bro). Metode cross-
# dataset yang sudah umum; BUKAN SFM. Didokumentasikan sebagai setup di paper.
ALIGN = {
  'duration' : ('Flow Duration',   'dur'),
  'fwd_pkts' : ('Tot Fwd Pkts',    'spkts'),
  'bwd_pkts' : ('Tot Bwd Pkts',    'dpkts'),
  'fwd_bytes': ('TotLen Fwd Pkts', 'sbytes'),
  'bwd_bytes': ('TotLen Bwd Pkts', 'dbytes'),
  'fwd_mean' : ('Fwd Pkt Len Mean','smean'),
  'bwd_mean' : ('Bwd Pkt Len Mean','dmean'),
  'src_load' : ('Flow Byts/s',     'sload'),
  'dst_load' : ('Bwd Pkts/s',      'dload'),
}
FEATS=list(ALIGN.keys())
print('9 fitur selaras (manual alignment):', FEATS)

## 0b. Lokasi data + PEMERIKSA PRASYARAT (hard-stop bila data tak ada)

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p): return p
    return None
UNSW_TRAIN=first_existing(['../../unswnb-15/data/UNSW_NB15_training-set.csv',
                           '../unswnb-15/data/UNSW_NB15_training-set.csv',
                           os.environ.get('UNSW_TRAIN')])
UNSW_TEST =first_existing(['../../unswnb-15/data/UNSW_NB15_testing-set.csv',
                           '../unswnb-15/data/UNSW_NB15_testing-set.csv',
                           os.environ.get('UNSW_TEST')])
CIC_PKL=first_existing(['../../CICDDoS2018/data/cleaned_100.pkl',
                        '../CICDDoS2018/data/cleaned_100.pkl',
                        'cleaned_100.pkl', os.environ.get('CIC_PKL')])
if CIC_PKL is None:
    key=os.environ.get('CIC_S3_KEY','unsw-far/CICDDoS2018/cleaned_100.pkl')
    try:
        import boto3; s3=boto3.client('s3',region_name=REGION)
        dst=os.path.join(OUTDIR,'cleaned_100.pkl')
        print('CIC tak ada lokal -> unduh s3://%s/%s'%(S3_BUCKET,key))
        s3.download_file(S3_BUCKET,key,dst); CIC_PKL=dst
    except Exception as e:
        print('(unduh CIC dari S3 gagal:',e,')')
print('UNSW_TRAIN:', UNSW_TRAIN)
print('UNSW_TEST :', UNSW_TEST)
print('CIC_PKL   :', CIC_PKL)
missing=[n for n,v in [('UNSW_TRAIN',UNSW_TRAIN),('UNSW_TEST',UNSW_TEST),('CIC_PKL',CIC_PKL)] if v is None]
assert not missing, ('PRASYARAT GAGAL - berkas tak ditemukan: %s. '
                     'Set path via env (UNSW_TRAIN/UNSW_TEST/CIC_PKL/CIC_S3_KEY) '
                     'atau jalankan di SageMaker tempat data tersedia.'%missing)
print('OK semua prasyarat data tersedia.')

## 1. Client-UNSW: terapkan alignment ke data UNSW SENDIRI

In [ ]:
def unsw_to_aligned(df):
    cols={k:v[1] for k,v in ALIGN.items()}
    miss=[cols[k] for k in FEATS if cols[k] not in df.columns]
    if miss: print('WARN kolom UNSW hilang:', miss)
    out=pd.DataFrame({k: pd.to_numeric(df.get(cols[k]), errors='coerce') for k in FEATS})
    out=out.replace([np.inf,-np.inf], np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    y=df['label'].astype(int).values
    return out.astype(float), y
utr=pd.read_csv(UNSW_TRAIN); ute=pd.read_csv(UNSW_TEST)
Xu_tr,yu_tr=unsw_to_aligned(utr); Xu_te,yu_te=unsw_to_aligned(ute)
print('UNSW train', Xu_tr.shape, 'pos', round(float(yu_tr.mean()),3),
      '| test', Xu_te.shape, 'pos', round(float(yu_te.mean()),3))

## 2. Client-CIC: muat (un-scale) + terapkan alignment ke data CIC SENDIRI

Kembalikan skala asli sebelum alignment; label biner attack=1/benign=0; split 70/30.

In [ ]:
import io
class _SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try: return super().find_class(module, name)
        except Exception: return type(name, (object,), {})
def _load_pkl(path):
    with open(path,'rb') as f:
        try: return pickle.load(f)
        except ModuleNotFoundError as e:
            print('[warn] load penuh gagal (%s); pakai SafeUnpickler'%e)
            f.seek(0); return _SafeUnpickler(f).load()
def cic_to_aligned(df):
    cols={k:v[0] for k,v in ALIGN.items()}
    miss=[cols[k] for k in FEATS if cols[k] not in df.columns]
    if miss: print('WARN kolom CIC hilang:', miss)
    out=pd.DataFrame({k: pd.to_numeric(df.get(cols[k]), errors='coerce') for k in FEATS})
    out=out.replace([np.inf,-np.inf],np.nan).fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float)
d=_load_pkl(CIC_PKL)
cic_feats=list(d['feature_names'])
X=np.asarray(d['X'],dtype=float)
scaler=d.get('scaler',None)
if scaler is not None and hasattr(scaler,'scale_') and hasattr(scaler,'mean_'):
    X=X*np.asarray(scaler.scale_,float)+np.asarray(scaler.mean_,float)
    print('CIC: un-scaled ke skala asli (scaler.mean_/scale_)')
else:
    print('CIC: scaler tak tersedia -> X apa adanya (periksa interpretasi!)')
cic_df=pd.DataFrame(X,columns=cic_feats)
y_raw=np.asarray(d['y'])
benign=None
for keyname in ('label_mapping','label_names','labels'):
    m=d.get(keyname)
    if isinstance(m,dict):
        for k,v in m.items():
            if str(k).lower().startswith('benign'): benign=v
    elif isinstance(m,(list,tuple)):
        for i,v in enumerate(m):
            if str(v).lower().startswith('benign'): benign=i
if benign is None: benign=0
yc_all=(y_raw!=benign).astype(int)
print('CIC benign-code dipakai:', benign, '| pos-rate', round(float(yc_all.mean()),3))
from sklearn.model_selection import train_test_split
Xc_all=cic_to_aligned(cic_df)
Xc_tr,Xc_te,yc_tr,yc_te=train_test_split(Xc_all,yc_all,test_size=0.3,random_state=SEED,stratify=yc_all)
print('CIC train (SEBELUM subsample)', Xc_tr.shape, 'test', Xc_te.shape)

# Subsample CIC train ke ~100k (stratified) agar rasio ke UNSW ~1.2:1 (hindari dominasi).
TARGET_CIC = 100_000
if len(Xc_tr) > TARGET_CIC:
    from sklearn.utils import resample
    idx_sub = resample(np.arange(len(Xc_tr)), n_samples=TARGET_CIC, replace=False,
                       stratify=yc_tr, random_state=SEED)
    Xc_tr = Xc_tr.iloc[idx_sub].reset_index(drop=True)
    yc_tr = yc_tr[idx_sub]
    print('CIC train (SETELAH subsample):', Xc_tr.shape,
          'pos-rate', round(float(yc_tr.mean()),3),
          '| rasio CIC:UNSW ~', round(len(Xc_tr)/len(Xu_tr),1), ':1')
else:
    print('CIC train sudah <= TARGET, tidak di-subsample')

## 3. Statistik per-klien (ANGKA NYATA) -> tabel dataset survey

In [ ]:
rows=[]
def stat(name, X, y, scheme):
    n=len(y); atk=int(np.sum(np.asarray(y)==1)); ben=int(np.sum(np.asarray(y)==0))
    rows.append({'client':name,'total':n,'attack':atk,'benign':ben,
                 'pos_rate':round(atk/n,4) if n else 0.0,'scheme':scheme})
stat('Client-UNSW (train)',Xu_tr,yu_tr,'per-dataset (UNSW release train)')
stat('Client-UNSW (test)', Xu_te,yu_te,'held-out UNSW test')
stat('Client-CIC (train)', Xc_tr,yc_tr,'per-dataset (CIC 70/30 seed)')
stat('Client-CIC (test)',  Xc_te,yc_te,'held-out CIC test')
dfst=pd.DataFrame(rows); dfst.to_csv(os.path.join(OUTDIR,'dataset_stats.csv'),index=False)
import IPython.display as ipd; ipd.display(dfst)
print('=== SEL 3 (statistik per-klien) SELESAI ===')

## 4. Simpan partisi per-klien (lokal + S3)

Nama `{cic,unsw}_train.npz` sesuai yang diharapkan node EC2 (`aws/fed_node.py`).

In [ ]:
def save_part(prefix, X, y):
    if X is None: return
    p=os.path.join(OUTDIR, prefix+'.npz')
    Xv=X.values if hasattr(X,'values') else np.asarray(X,float)
    np.savez_compressed(p, X=np.asarray(Xv,float), y=np.asarray(y,int), feats=np.array(FEATS))
    print('saved', p, np.asarray(Xv).shape)
for nm,Xv,yv in [('unsw_train',Xu_tr,yu_tr),('unsw_test',Xu_te,yu_te),
                 ('cic_train',Xc_tr,yc_tr),('cic_test',Xc_te,yc_te)]:
    save_part(nm, Xv, yv)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in os.listdir(OUTDIR):
        if f.endswith('.npz') or f=='dataset_stats.csv':
            sub='unsw' if f.startswith('unsw') else ('cic' if f.startswith('cic') else 'meta')
            s3.upload_file(os.path.join(OUTDIR,f),S3_BUCKET,f'{S3_PREFIX}/data/{sub}/{f}'); up+=1
    print('upload',up,'-> s3://%s/%s/data/'%(S3_BUCKET,S3_PREFIX))
except Exception as e: print('(upload S3 dilewati:',e,')')
print('=== SEL 4 (simpan partisi) SELESAI ===')

## 5. Catatan metodologi (untuk naskah survey)

- Penyelarasan fitur = **manual feature-name alignment** (padanan nama kolom antar-
  tool ekstraktor), metode cross-dataset yang **sudah ada** di literatur [F11].
  TIDAK memakai SFM. Diperlakukan sebagai *setup eksperimen*, bukan kontribusi.
- Kedua klien memakai tabel padanan yang sama tetapi menerapkannya pada datanya
  MASING-MASING -> tak ada pertukaran data mentah (privasi FL terjaga).
- CIC dikembalikan ke skala asli sebelum alignment; z-score **per-klien lokal**
  dilakukan di tahap training (konsisten deployment nyata).
- Keterbatasan alignment generik (fitur yang tak punya padanan dibuang) diakui
  jujur di Threats to Validity naskah survey.
- Data survey ditaruh di S3 prefix TERPISAH (`unsw-far/federated_survey/`) agar tak
  bercampur dengan data Paper 4 (`unsw-far/federated/`).